In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
import hashlib
import shutil

required={
    'solo_absolute_increment_ecdf.csv',
    'solo_activity_spells.csv',
    'solo_age_coverage.csv',
    'solo_age_distribution_table.csv',
    'solo_age_quantile_trajectories.csv',
    'solo_dropout_by_within_age_q_quintile.csv',
    'solo_dropout_events.csv',
    'solo_dropout_restart_by_age.csv',
    'solo_dropout_restart_duration_hazards.csv',
    'solo_ecdfs.csv',
    'solo_increment_ecdf.csv',
    'solo_increment_reference_fits.csv',
    'solo_increment_summary.csv',
    'solo_increment_symmetry.csv',
    'solo_metric_dictionary.csv',
    'solo_quantiled_moments.csv',
    'solo_restart_events.csv',
    'solo_spell_duration_distribution.csv',
    'solo_transition_counts.csv'
}
available={path.name for path in INPUT.glob('solo_*.csv')}
missing=required-available
if missing:
    raise FileNotFoundError(f'missing empirical inputs: {sorted(missing)}')


In [ ]:
schema={
    'solo_age_coverage.csv':{'age','scientists','zero_fraction'},
    'solo_age_distribution_table.csv':{'age','n','zero_fraction','positive_mean','positive_variance','p90','p99'},
    'solo_transition_counts.csv':{'transition','pairs','fraction'},
    'solo_increment_summary.csv':{'group','mean','variance'},
    'solo_increment_symmetry.csv':{'p_delta_zero','p_delta_positive','p_delta_negative'},
    'solo_dropout_restart_duration_hazards.csv':{'event','duration','hazard'},
    'solo_dropout_by_within_age_q_quintile.csv':{'q_quintile','probability'},
    'solo_ecdfs.csv':{'metric','x','cdf'}
}
for filename,columns in schema.items():
    frame=pd.read_csv(INPUT/filename,nrows=10)
    absent=columns-set(frame.columns)
    if absent:
        raise ValueError(f'{filename} missing columns: {sorted(absent)}')


In [ ]:
files=sorted(INPUT.glob('solo_*.csv'))
for path in files:
    shutil.copy2(path,OUTPUT/path.name)
if (INPUT/'Garden.png').exists():
    shutil.copy2(INPUT/'Garden.png',OUTPUT/'Garden.png')
raw_path=INPUT/CONFIG['raw_trajectory_file']
if raw_path.exists():
    hf.read_trajectory_matrix(raw_path,CONFIG['n_years'])
    shutil.copy2(raw_path,OUTPUT/raw_path.name)


In [ ]:
rows=[]
for path in sorted(OUTPUT.iterdir()):
    if path.is_file():
        rows.append({'file':path.name,'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
manifest=pd.DataFrame(rows)
manifest.to_csv(OUTPUT/'import_manifest.csv',index=False)
manifest
